## Transformer Block

In [8]:
import torch
import torch.nn as nn
from recap import Multi_Head_Attention


class Transformer_Block(nn.Module):

    def __init__(self, hidden_dim, context_size, dropout, head_nums):
        super().__init__()

        self.attn = Multi_Head_Attention(hidden_dim, context_size, dropout, head_nums)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim, bias=False),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim, bias=False),
        )

        self.layer_norm_1 = nn.LayerNorm(hidden_dim)
        self.layer_norm_2 = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)


    def forward(self, x, casual):
        
        x_res = x

        x = self.layer_norm_1(x)
        x, _ = self.attn(x, casual)
        x = self.dropout(x) + x_res

        x_res = x

        x = self.layer_norm_2(x)
        x = self.mlp(x)
        x = self.dropout(x) + x_res

        return x


## GPT Model

In [9]:
from recap import Embeddings


class GPT_Model(nn.Module):

    def __init__(self, hidden_dim, output_dim, context_size, dropout, head_nums, num_layers):
        super().__init__()

        self.emb_layer = Embeddings(vocab_size=output_dim, context_size=context_size, emb_dim=hidden_dim)
        self.emb_dropout = nn.Dropout(dropout)

        self.transformer_blocks = nn.ModuleList(
            [Transformer_Block(hidden_dim, context_size, dropout, head_nums)
             for _ in range(num_layers)]
        )
        
        self.output_layer = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, output_dim, bias=False)
        )

    def forward(self, x, casual):

        x = self.emb_dropout(self.emb_layer(x))
        
        for block in self.transformer_blocks:
            x = block(x, casual)

        x = self.output_layer(x)

        return x

## Final Outputs

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("cl100k_base")
vocab_size = len(tokenizer._mergeable_ranks)
context_size = 256
emb_dim = 512
hidden_dim = emb_dim
dropout = 0.2
head_nums = 8
num_layers = 2
casual = True

In [11]:
embedding = Embeddings(vocab_size, context_size, emb_dim)
text = "Hello, do you like tea? In the sunlit terraces of the palace."
text_tensor = torch.tensor(tokenizer.encode(text)).unsqueeze(0)

In [ ]:
gpt = GPT_Model(hidden_dim, output_dim=vocab_size, context_size=context_size, 
                dropout=dropout, head_nums=head_nums, num_layers=num_layers)

output = gpt(text_tensor, casual)
output.shape

torch.Size([1, 17, 100256])